In [ ]:
import os
import pandas as pd
from urllib.parse import urlparse

manifest = video_df.copy()

def clean_url(value):
    match = re.search(r"\]\((https?://[^)]+)\)", str(value))
    if match:
        return match.group(1)
    return str(value).strip()

manifest["clean_url"] = manifest["url"].apply(clean_url)
manifest["domain"] = manifest["clean_url"].apply(
    lambda x: urlparse(x).netloc.lower()
)

manifest["video_id"] = manifest["video_id"].astype(str).str.zfill(5)
manifest["signer_id"] = manifest["signer_id"].astype(str)

# Keep only direct MP4 links from SignStock
manifest = manifest[
    (manifest["domain"] == "signstock.blob.core.windows.net")
    & manifest["clean_url"].str.lower().str.endswith(".mp4")
].drop_duplicates(subset=["video_id"])

# Count videos and signers per gloss
gloss_stats = manifest.groupby("gloss").agg(
    video_count=("video_id", "nunique"),
    signer_count=("signer_id", "nunique")
).reset_index()

eligible_glosses = gloss_stats[
    (gloss_stats["video_count"] >= 20)
    & (gloss_stats["signer_count"] >= 3)
]["gloss"]

manifest = manifest[manifest["gloss"].isin(eligible_glosses)]

# Prioritize a manageable first batch
existing_ids = {
    os.path.splitext(f)[0]
    for f in os.listdir(VIDEO_PATH)
    if f.lower().endswith(".mp4")
}

manifest = manifest[~manifest["video_id"].isin(existing_ids)]

print("Eligible glosses:", manifest["gloss"].nunique())
print("Remaining candidate videos:", len(manifest))
print("\nVideos per gloss:")
print(
    manifest.groupby("gloss")["video_id"]
    .nunique()
    .sort_values(ascending=False)
    .head(15)
)

In [ ]:
print("All annotation rows:", len(video_df))

print("\nURL examples:")
print(video_df["url"].head(5).to_list())

print("\nURL domain counts:")
print(
    video_df["url"]
    .apply(lambda x: urlparse(clean_url(x)).netloc.lower())
    .value_counts()
    .head(15)
)

print("\nSignStock URL endings:")
signstock_urls = video_df[
    video_df["url"].apply(
        lambda x: "signstock.blob.core.windows.net" in clean_url(x).lower()
    )
]["url"].apply(clean_url)

print(signstock_urls.str.lower().str.extract(r"(\.[a-z0-9]+(?:\?.*)?$)")[0].value_counts())

In [ ]:
# Rebuild the SignStock subset
signstock_df = video_df.copy()

signstock_df["clean_url"] = signstock_df["url"].apply(clean_url)
signstock_df["domain"] = signstock_df["clean_url"].apply(
    lambda x: urlparse(x).netloc.lower()
)

signstock_df = signstock_df[
    (signstock_df["domain"] == "signstock.blob.core.windows.net")
    & signstock_df["clean_url"].str.lower().str.endswith(".mp4")
].copy()

signstock_df["video_id"] = (
    signstock_df["video_id"].astype(str).str.zfill(5)
)
signstock_df["signer_id"] = signstock_df["signer_id"].astype(str)

signstock_stats = signstock_df.groupby("gloss").agg(
    video_count=("video_id", "nunique"),
    signer_count=("signer_id", "nunique")
).reset_index()

print("Unique SignStock videos:", signstock_df["video_id"].nunique())
print("Unique glosses:", signstock_df["gloss"].nunique())

print("\nTop glosses by number of videos:")
print(
    signstock_stats.sort_values("video_count", ascending=False)
    .head(20).to_string(index=False)
)

print("\nGlosses with at least 3 videos and 3 signers:")
print(
    signstock_stats[
        (signstock_stats["video_count"] >= 3)
        & (signstock_stats["signer_count"] >= 3)
    ].sort_values("video_count", ascending=False)
    .head(20).to_string(index=False)
)

In [ ]:
pilot_glosses = ["champion", "later", "thin"]

pilot_df = signstock_df[
    signstock_df["gloss"].isin(pilot_glosses)
].copy()

pilot_df = pilot_df.drop_duplicates(subset=["video_id"])

print(
    pilot_df[
        ["gloss", "video_id", "signer_id", "clean_url"]
    ].sort_values(["gloss", "signer_id", "video_id"])
    .to_string(index=False)
)

print("\nSummary:")
print(
    pilot_df.groupby("gloss").agg(
        videos=("video_id", "nunique"),
        signers=("signer_id", "nunique")
    )
)

In [ ]:
import os
import re
import requests
from urllib.parse import urlparse

os.makedirs(VIDEO_PATH, exist_ok=True)

def extract_url(value):
    match = re.search(r"\]\((https?://[^)]+)\)", str(value))
    if match:
        return match.group(1)
    return str(value).strip()

download_results = []

for _, row in pilot_df.iterrows():
    video_id = str(row["video_id"]).zfill(5)
    video_url = extract_url(row["clean_url"])
    save_path = os.path.join(VIDEO_PATH, f"{video_id}.mp4")

    try:
        if os.path.exists(save_path) and os.path.getsize(save_path) > 0:
            status = "Already exists"
        else:
            response = requests.get(
                video_url,
                stream=True,
                timeout=30
            )
            response.raise_for_status()

            with open(save_path, "wb") as file:
                for chunk in response.iter_content(chunk_size=1024 * 256):
                    if chunk:
                        file.write(chunk)

            status = "Downloaded"

    except Exception as e:
        status = f"Failed: {str(e)[:100]}"

    download_results.append({
        "video_id": video_id,
        "gloss": row["gloss"],
        "signer_id": row["signer_id"],
        "status": status
    })

download_df = pd.DataFrame(download_results)
print(download_df.to_string(index=False))

print("\nDownload summary:")
print(download_df["status"].value_counts())


In [ ]:
import os
import cv2
import pandas as pd

check_results = []

for _, row in pilot_df.iterrows():
    video_id = str(row["video_id"]).zfill(5)
    path = os.path.join(VIDEO_PATH, f"{video_id}.mp4")

    exists = os.path.exists(path)
    size_mb = (
        round(os.path.getsize(path) / (1024 * 1024), 2)
        if exists else 0
    )

    readable = False
    frame_count = 0

    if exists and size_mb > 0:
        cap = cv2.VideoCapture(path)
        readable = cap.isOpened()

        if readable:
            frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            readable = frame_count > 0

        cap.release()

    check_results.append({
        "video_id": video_id,
        "gloss": row["gloss"],
        "signer_id": row["signer_id"],
        "size_MB": size_mb,
        "frames": frame_count,
        "readable": readable
    })

check_df = pd.DataFrame(check_results)
print(check_df.to_string(index=False))

print("\nReadable videos:", check_df["readable"].sum(), "of", len(check_df))

In [ ]:
import os
import cv2
import matplotlib.pyplot as plt

sample_glosses = ["champion", "later", "thin"]

fig, axes = plt.subplots(1, 3, figsize=(12, 4))

for ax, gloss in zip(axes, sample_glosses):
    row = pilot_df[pilot_df["gloss"] == gloss].iloc[0]
    video_id = str(row["video_id"]).zfill(5)

    path = os.path.join(VIDEO_PATH, f"{video_id}.mp4")
    cap = cv2.VideoCapture(path)

    success, frame = cap.read()
    cap.release()

    if success:
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        ax.imshow(frame)
        ax.set_title(f"{gloss} | ID: {video_id}")
    else:
        ax.set_title(f"{gloss} | Frame unavailable")

    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
signer_summary = (
    pilot_df.groupby("gloss")
    .agg(
        total_videos=("video_id", "nunique"),
        unique_signers=("signer_id", "nunique"),
        signer_list=("signer_id", lambda x: sorted(set(x.astype(str))))
    )
    .reset_index()
)

print(signer_summary.to_string(index=False))

print("\nTotal unique signers:",
      pilot_df["signer_id"].nunique())

In [ ]:
signer_class_table = pd.crosstab(
    pilot_df["signer_id"].astype(str),
    pilot_df["gloss"]
)

print(signer_class_table)

print("\nVideos per signer:")
print(
    pilot_df.groupby("signer_id")["video_id"]
    .nunique()
    .sort_values(ascending=False)
)

In [ ]:
train_signers = ["4", "39"]

pilot_df["signer_id"] = pilot_df["signer_id"].astype(str)

train_df = pilot_df[
    pilot_df["signer_id"].isin(train_signers)
].copy()

test_df = pilot_df[
    ~pilot_df["signer_id"].isin(train_signers)
].copy()

print("TRAINING SET")
print(train_df.groupby("gloss").agg(
    videos=("video_id", "nunique"),
    signers=("signer_id", "nunique")
))

print("\nTESTING SET")
print(test_df.groupby("gloss").agg(
    videos=("video_id", "nunique"),
    signers=("signer_id", "nunique")
))

print("\nTraining signers:", sorted(train_df["signer_id"].unique()))
print("Testing signers:", sorted(test_df["signer_id"].unique()))

print("\nGlosses missing from test:")
print(sorted(set(train_df["gloss"]) - set(test_df["gloss"])))

In [ ]:
import numpy as np
import os

X_train = []
y_train = []
X_test = []
y_test = []

for _, row in train_df.iterrows():
    video_id = str(row["video_id"]).zfill(5)
    path = os.path.join(VIDEO_PATH, f"{video_id}.mp4")

    frames = preprocess_video(path, num_frames=8, image_size=224)

    if frames is not None:
        X_train.append(frames)
        y_train.append(row["gloss"])

for _, row in test_df.iterrows():
    video_id = str(row["video_id"]).zfill(5)
    path = os.path.join(VIDEO_PATH, f"{video_id}.mp4")

    frames = preprocess_video(path, num_frames=8, image_size=224)

    if frames is not None:
        X_test.append(frames)
        y_test.append(row["gloss"])

X_train = np.array(X_train, dtype=np.float32)
X_test = np.array(X_test, dtype=np.float32)

y_train = np.array(y_train)
y_test = np.array(y_test)

print("Training video tensor:", X_train.shape)
print("Testing video tensor:", X_test.shape)
print("Training labels:", y_train.shape)
print("Testing labels:", y_test.shape)

In [ ]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

print("Class mapping:")

for index, gloss in enumerate(label_encoder.classes_):
    print(f"{gloss} -> {index}")

print("\nEncoded training labels:", y_train_encoded)
print("Encoded testing labels:", y_test_encoded)

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

tf.random.set_seed(42)
np.random.seed(42)

model = models.Sequential([
    layers.Input(shape=(8, 224, 224, 3)),

    layers.Conv3D(16, kernel_size=(3, 3, 3), activation="relu"),
    layers.MaxPooling3D(pool_size=(1, 2, 2)),

    layers.Conv3D(32, kernel_size=(3, 3, 3), activation="relu"),
    layers.GlobalAveragePooling3D(),

    layers.Dense(32, activation="relu"),
    layers.Dropout(0.3),

    layers.Dense(3, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

In [ ]:
history = model.fit(
    X_train,
    y_train_encoded,
    epochs=10,
    batch_size=2,
    validation_data=(X_test, y_test_encoded),
    verbose=1
)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Test Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Model Accuracy")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Test Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Model Loss")
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

predictions = model.predict(X_test)

predicted_labels = np.argmax(predictions, axis=1)

actual_glosses = label_encoder.inverse_transform(y_test_encoded)
predicted_glosses = label_encoder.inverse_transform(predicted_labels)

results_df = pd.DataFrame({
    "Actual Sign": actual_glosses,
    "Predicted Sign": predicted_glosses,
    "Correct": actual_glosses == predicted_glosses
})

print(results_df.to_string(index=False))

print("\nClassification Report:")
print(
    classification_report(
        y_test_encoded,
        predicted_labels,
        labels=[0, 1, 2],
        target_names=label_encoder.classes_,
        zero_division=0
    )
)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    y_test_encoded,
    predicted_labels,
    labels=[0, 1, 2]
)

plt.figure(figsize=(6, 5))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=label_encoder.classes_,
    yticklabels=label_encoder.classes_
)

plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
import os
import json

os.makedirs(OUTPUT_PATH, exist_ok=True)

model_path = os.path.join(OUTPUT_PATH, "asl_baseline_model.keras")
model.save(model_path)

results_path = os.path.join(OUTPUT_PATH, "asl_baseline_predictions.csv")
results_df.to_csv(results_path, index=False)

mapping_path = os.path.join(OUTPUT_PATH, "asl_label_mapping.json")

with open(mapping_path, "w") as file:
    json.dump(
        {
            str(index): gloss
            for index, gloss in enumerate(label_encoder.classes_)
        },
        file,
        indent=4
    )

print("Saved model:", model_path)
print("Saved predictions:", results_path)
print("Saved label mapping:", mapping_path)

In [ ]:


full_stats = (
    video_df.groupby("gloss")
    .agg(
        total_videos=("video_id", "nunique"),
        unique_signers=("signer_id", "nunique")
    )
    .reset_index()
)

print("Total annotated videos:", video_df["video_id"].nunique())
print("Total glosses:", video_df["gloss"].nunique())

print("\nGlosses with at least 10 videos and 5 signers:")

eligible_glosses = full_stats[
    (full_stats["total_videos"] >= 10) &
    (full_stats["unique_signers"] >= 5)
].sort_values(
    ["unique_signers", "total_videos"],
    ascending=False
)

print("Number of eligible glosses:", len(eligible_glosses))
print(eligible_glosses.head(25).to_string(index=False))

In [ ]:
import re
from urllib.parse import urlparse

def extract_clean_url(value):
    match = re.search(r"\]\((https?://[^)]+)\)", str(value))
    if match:
        return match.group(1)
    return str(value).strip()

candidate_df = video_df.copy()

candidate_df["clean_url"] = candidate_df["url"].apply(
    extract_clean_url
)

candidate_df["domain"] = candidate_df["clean_url"].apply(
    lambda url: urlparse(url).netloc.lower()
)

candidate_df["is_mp4_url"] = candidate_df["clean_url"].apply(
    lambda url: urlparse(url).path.lower().endswith(".mp4")
)

# Keep URLs that look like direct MP4 video links
direct_video_df = candidate_df[
    candidate_df["is_mp4_url"]
].copy()

# Remove records without a usable signer ID
direct_video_df = direct_video_df[
    direct_video_df["signer_id"].notna()
].copy()

direct_stats = (
    direct_video_df.groupby("gloss")
    .agg(
        videos=("video_id", "nunique"),
        signers=("signer_id", "nunique")
    )
    .reset_index()
)

eligible_direct = direct_stats[
    (direct_stats["videos"] >= 10) &
    (direct_stats["signers"] >= 5)
].sort_values(
    ["signers", "videos"],
    ascending=False
)

print("Records with .mp4 URLs:", len(direct_video_df))
print("Unique glosses with .mp4 URLs:",
      direct_video_df["gloss"].nunique())

print("\nMP4 URL counts by source:")
print(
    direct_video_df["domain"]
    .value_counts()
    .head(15)
    .to_string()
)

print("\nGlosses meeting our threshold using MP4 URLs:")
print("Eligible glosses:", len(eligible_direct))
print(eligible_direct.head(25).to_string(index=False))


In [ ]:

# Step 64: Prepare a larger candidate dataset

import pandas as pd
import re
from urllib.parse import urlparse

# Select five candidate ASL signs
selected_glosses = ["basketball", "drink", "tall", "cold", "write"]

# Keep only the selected glosses
experiment_df = candidate_df[
    candidate_df["gloss"].isin(selected_glosses)
].copy()

# Clean signer IDs
experiment_df["signer_id"] = (
    experiment_df["signer_id"].astype(str).str.strip()
)

invalid_signers = ["", "nan", "none", "-1", "unknown"]
experiment_df = experiment_df[
    ~experiment_df["signer_id"].str.lower().isin(invalid_signers)
].copy()

# Remove duplicate videos
experiment_df = experiment_df.drop_duplicates(
    subset=["gloss", "video_id"]
)

# Summarize videos and signers for each class
class_summary = (
    experiment_df.groupby("gloss")
    .agg(
        videos=("video_id", "nunique"),
        signers=("signer_id", "nunique")
    )
    .reset_index()
)

print("CLASS SUMMARY")
print(class_summary.to_string(index=False))

print("\nVIDEO SOURCES FOR EACH CLASS")
source_summary = (
    experiment_df.groupby(["gloss", "domain"])
    .agg(videos=("video_id", "nunique"))
    .reset_index()
)

print(source_summary.to_string(index=False))

print("\nTOTAL UNIQUE VIDEOS:", experiment_df["video_id"].nunique())
print("TOTAL DISTINCT SIGNERS:", experiment_df["signer_id"].nunique())


In [ ]:

# Step 64: Prepare a larger candidate dataset

import pandas as pd
import re
from urllib.parse import urlparse

# Select five candidate ASL signs
selected_glosses = ["basketball", "drink", "tall", "cold", "write"]

# Keep only the selected glosses
experiment_df = candidate_df[
    candidate_df["gloss"].isin(selected_glosses)
].copy()

# Clean signer IDs
experiment_df["signer_id"] = (
    experiment_df["signer_id"].astype(str).str.strip()
)

invalid_signers = ["", "nan", "none", "-1", "unknown"]
experiment_df = experiment_df[
    ~experiment_df["signer_id"].str.lower().isin(invalid_signers)
].copy()

# Remove duplicate videos
experiment_df = experiment_df.drop_duplicates(
    subset=["gloss", "video_id"]
)

# Summarize videos and signers for each class
class_summary = (
    experiment_df.groupby("gloss")
    .agg(
        videos=("video_id", "nunique"),
        signers=("signer_id", "nunique")
    )
    .reset_index()
)

print("CLASS SUMMARY")
print(class_summary.to_string(index=False))

print("\nVIDEO SOURCES FOR EACH CLASS")
source_summary = (
    experiment_df.groupby(["gloss", "domain"])
    .agg(videos=("video_id", "nunique"))
    .reset_index()
)

print(source_summary.to_string(index=False))

print("\nTOTAL UNIQUE VIDEOS:", experiment_df["video_id"].nunique())
print("TOTAL DISTINCT SIGNERS:", experiment_df["signer_id"].nunique())


In [ ]:

import requests
from urllib.parse import urlparse
import re
import pandas as pd

def get_clean_url(value):
    match = re.search(r"\]\((https?://[^)]+)\)", str(value))
    return match.group(1) if match else str(value).strip()

# Prefer direct MP4 links for the first test
test_df = experiment_df.copy()
test_df["clean_url"] = test_df["url"].apply(get_clean_url)

test_df["is_direct_mp4"] = test_df["clean_url"].apply(
    lambda url: urlparse(url).path.lower().endswith(".mp4")
)

# Test up to 3 direct MP4 links per class
sample_df = (
    test_df[test_df["is_direct_mp4"]]
    .groupby("gloss", group_keys=False)
    .head(3)
    .copy()
)

def test_url(url):
    try:
        response = requests.get(
            url,
            headers={"Range": "bytes=0-1023"},
            timeout=15,
            stream=True,
            allow_redirects=True
        )
        status = response.status_code
        content_type = response.headers.get("content-type", "")
        response.close()

        accessible = status in [200, 206]
        return pd.Series({
            "http_status": status,
            "content_type": content_type,
            "accessible": accessible
        })

    except Exception as error:
        return pd.Series({
            "http_status": "ERROR",
            "content_type": str(error)[:100],
            "accessible": False
        })

results = sample_df["clean_url"].apply(test_url)
sample_df = pd.concat([sample_df, results], axis=1)

print("URL ACCESSIBILITY TEST")
print(
    sample_df[
        ["gloss", "video_id", "signer_id",
         "domain", "http_status", "accessible"]
    ].to_string(index=False)
)

print("\nACCESSIBLE URL COUNTS")
print(
    sample_df.groupby("gloss")["accessible"]
    .agg(["sum", "count"])
    .to_string()
)


In [ ]:

import os
import cv2
import requests
import pandas as pd

DOWNLOAD_PATH = "/content/asl_dataset/expanded_sample"
os.makedirs(DOWNLOAD_PATH, exist_ok=True)

download_records = []

for _, row in sample_df.iterrows():
    gloss = row["gloss"]
    video_id = str(row["video_id"])
    url = row["clean_url"]

    # Use a separate filename for each gloss and video
    filename = f"{gloss}_{video_id}.mp4"
    filepath = os.path.join(DOWNLOAD_PATH, filename)

    record = {
        "gloss": gloss,
        "video_id": video_id,
        "signer_id": str(row["signer_id"]),
        "filepath": filepath,
        "downloaded": False,
        "readable": False,
        "frame_count": 0
    }

    try:
        response = requests.get(url, timeout=30)
        response.raise_for_status()

        with open(filepath, "wb") as file:
            file.write(response.content)

        record["downloaded"] = os.path.getsize(filepath) > 0

        if record["downloaded"]:
            cap = cv2.VideoCapture(filepath)
            record["readable"] = cap.isOpened()
            record["frame_count"] = int(
                cap.get(cv2.CAP_PROP_FRAME_COUNT)
            )
            cap.release()

            if record["frame_count"] <= 0:
                record["readable"] = False

    except Exception as error:
        record["error"] = str(error)[:150]

    download_records.append(record)

download_report = pd.DataFrame(download_records)

print("DOWNLOAD AND VIDEO VALIDATION REPORT")
print(
    download_report[
        ["gloss", "video_id", "downloaded",
         "readable", "frame_count"]
    ].to_string(index=False)
)

print("\nSUMMARY")
print("Videos downloaded:", int(download_report["downloaded"].sum()))
print("Videos readable:", int(download_report["readable"].sum()))
print("Videos attempted:", len(download_report))


In [ ]:




import os
import cv2

failed_df = download_report[
    ~download_report["readable"]
].copy()

for _, row in failed_df.iterrows():
    path = row["filepath"]

    print("\nGloss:", row["gloss"])
    print("Video ID:", row["video_id"])
    print("File exists:", os.path.exists(path))

    if os.path.exists(path):
        print("File size:", os.path.getsize(path), "bytes")

        with open(path, "rb") as file:
            header = file.read(200)

        print("First bytes:", repr(header[:100]))

        cap = cv2.VideoCapture(path)
        print("OpenCV opened:", cap.isOpened())
        cap.release()

    print("-" * 40)
